# FOI – 230512035 – Number of Private Law Residence Orders Made

## 1. Import packages and set options 
<a name="import_packages"></a>

In [ ]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

#import the pacakges required to produce an excel spreadsheet
import xlsxwriter
import boto3, io, os, pathlib

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [ ]:
#this is the database we will be extracting from
database = "familyman_dev_v3"
derived = "familyman_derived_dev_v3"

#this extracts the February snapshot from athena
snapshot_date = "2023-02-09"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#setting current year and quarter
pub_quarter = 4
pub_year = 2022

## Extract applicant information

In [ ]:
#Joined three familyman_dev_v3 tables together to extract case_number, role_model, role, field_model, and value columns to help produce the final outputs
#Filtered using the snapshot_date = "2023-05-10" across all three tables and filtered for b.ROLE_MODEL = 'APLC and a.FIELD_MODEL = 'APLC_RC' 
pydb.create_temp_table(
f""" 
SELECT c.case_number, 
    b.role_model, 
    a.role, 
    a.field_model, 
    a.value

FROM {database}.role_fields a
LEFT JOIN {database}.field_models b
ON a.field_model = b.field_model
LEFT JOIN {database}.roles c
ON a.role = c.role

WHERE a.FIELD_MODEL = 'APLC_RC' 
AND b.ROLE_MODEL = 'APLC' 
AND a.mojap_snapshot_date = DATE('{snapshot_date}')
AND b.mojap_snapshot_date = DATE('{snapshot_date}')
AND c.mojap_snapshot_date = DATE('{snapshot_date}');
""",

"applicant_data")

In [ ]:
applicant_data = pydb.read_sql_query("SELECT * FROM __temp__.applicant_data")
applicant_data

## Residence orders - *FCSQ Check*

In [ ]:
#Counts the number of residence orders made from 2011 to 2022
#Grouped by year
#Filters for private law, orders made, and section 8 residence orders 
pydb.create_temp_table(
f""" 
SELECT year, 
'30 Section 8 Residence Order' as order_type,
COUNT(*) as Count

FROM fcsq.ca_all_disposals

WHERE disp_type = 'Order made'
AND order_desc = 'Section 8 Residence Order'
AND order_case_type = 'P'
AND year > 2010
AND year < 2023

GROUP BY year
""",

"FCSQ_check")

In [ ]:
FCSQ_check = pydb.read_sql_query("SELECT * FROM __temp__.FCSQ_check")
FCSQ_check

## Residence orders by region 

In [ ]:
pydb.create_temp_table(
f""" 
SELECT DISTINCT A.year,
A.quarter, 
A.disp_date,
A.case_number,
cast((A.EVENT / 100000000) as int) as event_code,
A.event,
'30 Section 8 Residence Order' as order_type,
B.region,
B.region_pre2014

FROM fcsq.ca_all_disposals A
LEFT JOIN fcsq.court_mv_feb21_dfj AS B
      ON cast((A.EVENT / 100000000) as int) = B.CODE

WHERE disp_type = 'Order made'
AND order_desc = 'Section 8 Residence Order'
AND order_case_type = 'P'
AND year > 2010 

""",

"residence_orders")

In [ ]:
residence_orders_data = pydb.read_sql_query("SELECT * FROM __temp__.residence_orders")
residence_orders_data

## Residence orders relationship to child (RTC)

In [ ]:
pydb.create_temp_table(
f""" 
SELECT A.year,
A.quarter, 
A.disp_date,
A.case_number,
event_code,
A.event,
order_type,
B.value as RTC,
B.role

FROM __temp__.residence_orders A
LEFT JOIN __temp__.applicant_data AS B
      ON A.case_number = B.case_number
""",

"residence_orders_RTC")

In [ ]:
residence_orders_RTC_data = pydb.read_sql_query("SELECT * FROM __temp__.residence_orders_RTC")
residence_orders_RTC_data

## *Check for all the different types of RTC*

In [ ]:
RTC = pydb.read_sql_query("SELECT DISTINCT RTC FROM __temp__.residence_orders_RTC")
RTC

## RTC Catogories

In [ ]:
pydb.create_temp_table(
f""" 
SELECT
  A.*,
  CASE WHEN RTC IN ('Father','Mother') THEN 'Parent'
       WHEN RTC IN ('Grandparent', 'Special Guardian', 'Other') THEN RTC
       WHEN RTC IN ('Step father', 'Step mother') THEN 'Step parent'
       WHEN RTC IN ('','--','Child') THEN 'Not recorded'/*Records where the RTC is listed as 'Child' - have added these to other as it would be an error?*/
       WHEN RTC IS NULL THEN 'Not recorded'
    ELSE 'Check'
   END AS RTC_GROUP,
  B.REGION
FROM
   __temp__.residence_orders_RTC AS A
     LEFT JOIN fcsq.court_mv_feb21_dfj AS B
      ON A.EVENT_CODE = B.CODE;
""",

"RTC_CATS")

In [ ]:
RTC_CATS = pydb.read_sql_query("SELECT * FROM __temp__.RTC_CATS")
RTC_CATS

## Distinct RTC Catogories 

In [ ]:
pydb.create_temp_table(
f""" 
SELECT
DISTINCT Year,
    Quarter,
    disp_date,
    Case_Number,
    Event_Code,
    Event,
    RTC_GROUP,
    Region
 FROM
   __temp__.RTC_CATS;
""",

"RTC_DISTINCT_CATS")

In [ ]:
RTC_DISTINCT_CATS = pydb.read_sql_query("SELECT * FROM __temp__.RTC_DISTINCT_CATS")
RTC_DISTINCT_CATS

## *Check there are no 'checks' in the RTC grouping* 

In [ ]:
RTC_CATS_check = pydb.read_sql_query("SELECT DISTINCT RTC FROM __temp__.RTC_CATS")
RTC_CATS_check

## *Check all regions* 

In [ ]:
RTC_CATS_region_check = pydb.read_sql_query("SELECT DISTINCT region FROM __temp__.RTC_CATS")
RTC_CATS_region_check

In [ ]:
A = pydb.read_sql_query("SELECT YEAR, count(*) as count FROM __temp__.RTC_CATS WHERE REGION = 'NON' GROUP BY YEAR;")
A

In [ ]:
B = pydb.read_sql_query("SELECT DISTINCT EVENT_CODE, REGION_PRE2014 FROM __temp__.Residence_orders WHERE REGION = 'NON' and REGION_PRE2014 = 'WALES';")
B

In [ ]:
C = pydb.read_sql_query("SELECT DISTINCT REGION FROM __temp__.Residence_orders WHERE EVENT_CODE IN (401,478);")
C

## Total Orders

In [ ]:
pydb.create_temp_table(
f""" 
SELECT
  YEAR,
  COUNT(*) AS COUNT
FROM
  __temp__.residence_orders
WHERE
  REGION <> 'WALES'
  AND EVENT_CODE NOT IN (401,478)
GROUP BY
  YEAR;
""",

"TOTAL_ORDERS")

In [ ]:
TOTAL_ORDERS = pydb.read_sql_query("SELECT * FROM __temp__.TOTAL_ORDERS;")
TOTAL_ORDERS

## Total Applicants

In [ ]:
pydb.create_temp_table(
f""" 
SELECT
  YEAR,
  RTC_GROUP,
  COUNT (*) AS COUNT
FROM
  __temp__.RTC_DISTINCT_CATS
WHERE
  REGION <> 'WALES'
  AND EVENT_CODE NOT IN (401,478) 
GROUP BY
  YEAR,
  RTC_GROUP,
  REGION;
""",

"TOTAL_APPLICANTS")

In [ ]:
TOTAL_APPLICANTS = pydb.read_sql_query("SELECT * FROM __temp__.TOTAL_APPLICANTS;")
TOTAL_APPLICANTS

## Create Final Output

In [ ]:
#creating a panda dataframe for both order and applicant tables
#uses the queries underneath both order and applicant table
df1 = pd.DataFrame(TOTAL_ORDERS)

df2 = pd.DataFrame(TOTAL_APPLICANTS)

In [ ]:
#creates a seperate sheet for each dataframe 
with io.BytesIO() as xlsx_data:
    with pd.ExcelWriter(xlsx_data) as writer:
        
        df1.to_excel(writer, sheet_name='orders', index=False)

        df2.to_excel(writer, sheet_name='applicants', index=False)

    ## Upload the in-memory data to S3
    s3 = boto3.resource("s3")
    bucket = s3.Bucket("alpha-family-data")
    object_name = "FOIs/FOI 230512035 – working table.xlsx"
    s3.Bucket("alpha-family-data").put_object(Key=object_name, Body=xlsx_data.getvalue())